In [ ]:
# Gerekli kütüphaneler
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
from scipy.stats import poisson

## Bölüm 2: Python ile Sayısal (Numerical) MLE

Poisson dağılımı için Negative Log-Likelihood fonksiyonu minimize edilerek λ (lambda) parametresi bulunacaktır.

In [ ]:
# Gözlemlenen trafik verisi (1 dakikada geçen araç sayısı)
traffic_data = np.array([12, 15, 10, 8, 14, 11, 13, 16, 9, 12, 11, 14, 10, 15])

def negative_log_likelihood(lam, data):
    """
    Poisson dağılımı için Negatif Log-Likelihood hesaplar.
    
    Türetme:
      log L(λ) = -n*λ + (Σk_i)*log(λ) - Σlog(k_i!)
      NLL      =  n*λ - (Σk_i)*log(λ)   (log(k!) sabiti ihmal edilir)
    """
    l = lam[0]          # scipy minimize lambda'yı dizi olarak verir
    n = len(data)
    nll = n * l - np.sum(data) * np.log(l)   # negatif log-likelihood
    return nll

# Başlangıç tahmini (optimizer buradan aramaya başlar)
initial_guess = [1.0]

# Optimizasyon: NLL'yi minimize etmek = Likelihood'u maximize etmek
# bounds ile λ > 0 sınırı uygulanır (Poisson için negatif λ anlamsız)
result = opt.minimize(
    negative_log_likelihood,
    initial_guess,
    args=(traffic_data,),
    bounds=[(0.001, None)]
)

lambda_mle      = result.x[0]             # sayısal MLE tahmini
lambda_analytic = np.mean(traffic_data)   # analitik çözüm (ortalama)

print(f"Sayısal Tahmin  (MLE lambda) : {lambda_mle:.4f}")
print(f"Analitik Tahmin (Ortalama)   : {lambda_analytic:.4f}")

## Bölüm 3: Model Karşılaştırma ve Görselleştirme

Bulunan λ değeriyle Poisson PMF grafiği çizilecek, üzerine gerçek verinin histogramı eklenecektir.

In [ ]:
# Olasılık hesabı için k aralığı belirlenir
k_values  = np.arange(0, 30)
pmf_values = poisson.pmf(k_values, lambda_mle)   # Poisson PMF değerleri

fig, ax = plt.subplots(figsize=(10, 6))

# Gerçek verinin normalize edilmiş histogramı
bins = range(min(traffic_data), max(traffic_data) + 2)
ax.hist(
    traffic_data,
    bins=bins,
    density=True,         # normalize: toplam alan = 1
    alpha=0.6,
    color='steelblue',
    label='Gerçek Veri (Histogram)',
    align='left'
)

# Poisson PMF çubuk grafiği
ax.bar(
    k_values,
    pmf_values,
    alpha=0.7,
    color='orange',
    width=0.4,
    label=f'Poisson PMF  (λ = {lambda_mle:.2f})'
)

# Grafik düzeni
ax.set_xlabel('Araç Sayısı (k)')
ax.set_ylabel('Olasılık')
ax.set_title('Poisson MLE — Model ve Gerçek Veri Karşılaştırması')
ax.legend()
ax.set_xlim(0, 25)

plt.tight_layout()
plt.show()